# ABM Calibration Analysis
Price vs Fundamental Value · Return Distributions · Autocorrelation · Calm vs Stressed Regime Comparison

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.stats.stattools import durbin_watson
import warnings
warnings.filterwarnings('ignore')

CALM_PATH     = 'output/fundamental_value_calm.csv'
STRESSED_PATH = 'output/fundamental_value_stressed.csv'
PARAMS_PATH   = 'output/calibrated_params.csv'

calm     = pd.read_csv(CALM_PATH)
stressed = pd.read_csv(STRESSED_PATH)
params   = pd.read_csv(PARAMS_PATH)

print(params[['regime','gamma','beta','alpha','sigma_v','sigma_n','g','log_lik']].to_string(index=False))


## 1 · Price vs Fundamental Value

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=False)

for ax, df, regime, color_p, color_v in [
    (axes[0], calm,     'Calm',     '#2563eb', '#16a34a'),
    (axes[1], stressed, 'Stressed', '#dc2626', '#d97706'),
]:
    t = df['t']
    ax.plot(t, df['log_price'],   color=color_p, lw=1.4, label='Log Price $p_t$', zorder=3)
    ax.plot(t, df['v_smoothed'],  color=color_v, lw=1.4, label='Fundamental $V_t$ (smoothed)', zorder=3)
    ax.fill_between(t,
        df['v_smoothed'] - 1.96 * df['v_stderr'],
        df['v_smoothed'] + 1.96 * df['v_stderr'],
        color=color_v, alpha=0.15, label='95% posterior band')
    mispricing = df['log_price'] - df['v_smoothed']
    ax2 = ax.twinx()
    ax2.fill_between(t, mispricing, 0,
        where=(mispricing > 0), color='#ef4444', alpha=0.2, label='Overvalued')
    ax2.fill_between(t, mispricing, 0,
        where=(mispricing < 0), color='#3b82f6', alpha=0.2, label='Undervalued')
    ax2.axhline(0, color='grey', lw=0.5, ls='--')
    ax2.set_ylabel('Mispricing $p_t - V_t$', fontsize=9)
    ax.set_title(f'{regime} Regime', fontweight='bold')
    ax.set_ylabel('Log Price / Fundamental')
    ax.legend(loc='upper left', fontsize=8)
    ax2.legend(loc='upper right', fontsize=8)

axes[1].set_xlabel('Trading Day')
plt.suptitle('Price vs Kalman-Smoothed Fundamental Value', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('output/price_vs_fundamental.png', dpi=150, bbox_inches='tight')
plt.show()


## 2 · Mispricing Distribution

In [ ]:
calm_mis     = calm['log_price']     - calm['v_smoothed']
stressed_mis = stressed['log_price'] - stressed['v_smoothed']

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for ax, mis, regime, color in [
    (axes[0], calm_mis,     'Calm',     '#2563eb'),
    (axes[1], stressed_mis, 'Stressed', '#dc2626'),
]:
    ax.hist(mis, bins=30, color=color, alpha=0.7, density=True, edgecolor='white', lw=0.4)
    xr = np.linspace(mis.min(), mis.max(), 200)
    ax.plot(xr, stats.norm.pdf(xr, mis.mean(), mis.std()),
            'k--', lw=1.2, label='Normal fit')
    ax.axvline(0, color='grey', lw=0.8, ls=':')
    sk, ku = stats.skew(mis), stats.kurtosis(mis)
    _, jb_p = stats.jarque_bera(mis)
    ax.set_title(f'{regime}  |  skew={sk:.2f}  kurt={ku:.2f}  JB p={jb_p:.3f}',
                 fontweight='bold', fontsize=10)
    ax.set_xlabel('Mispricing $p_t - V_t$')
    ax.set_ylabel('Density')
    ax.legend(fontsize=8)

plt.suptitle('Mispricing Distribution: Calm vs Stressed', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('output/mispricing_distribution.png', dpi=150, bbox_inches='tight')
plt.show()


## 3 · Return Distributions (Log-Returns)

In [ ]:
calm_ret     = np.diff(calm['log_price'].values)
stressed_ret = np.diff(stressed['log_price'].values)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for ax, ret, regime, color in [
    (axes[0], calm_ret,     'Calm',     '#2563eb'),
    (axes[1], stressed_ret, 'Stressed', '#dc2626'),
]:
    ax.hist(ret, bins=35, color=color, alpha=0.7, density=True,
            edgecolor='white', lw=0.4)
    xr = np.linspace(ret.min(), ret.max(), 300)
    ax.plot(xr, stats.norm.pdf(xr, ret.mean(), ret.std()),
            'k--', lw=1.4, label='Normal')
    # Student-t fit
    df_t, loc_t, scale_t = stats.t.fit(ret)
    ax.plot(xr, stats.t.pdf(xr, df_t, loc_t, scale_t),
            color='orange', lw=1.4, ls='-.',
            label=f'Student-t (df={df_t:.1f})')
    sk, ku = stats.skew(ret), stats.kurtosis(ret)
    _, jb_p = stats.jarque_bera(ret)
    ax.set_title(f'{regime}  |  skew={sk:.2f}  kurt={ku:.2f}  JB p={jb_p:.3f}',
                 fontweight='bold', fontsize=10)
    ax.set_xlabel('Log Return')
    ax.set_ylabel('Density')
    ax.legend(fontsize=8)

plt.suptitle('Return Distribution: Calm vs Stressed', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('output/return_distribution.png', dpi=150, bbox_inches='tight')
plt.show()


## 4 · QQ Plots (Fat Tails Check)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, ret, regime, color in [
    (axes[0], calm_ret,     'Calm',     '#2563eb'),
    (axes[1], stressed_ret, 'Stressed', '#dc2626'),
]:
    (osm, osr), (slope, intercept, r) = stats.probplot(ret, dist='norm')
    ax.scatter(osm, osr, color=color, s=12, alpha=0.7, zorder=3)
    ax.plot(osm, slope * np.array(osm) + intercept,
            'k--', lw=1.2, label=f'Normal line  R²={r**2:.3f}')
    ax.set_title(f'QQ Plot — {regime}', fontweight='bold')
    ax.set_xlabel('Theoretical Quantiles')
    ax.set_ylabel('Sample Quantiles')
    ax.legend(fontsize=8)

plt.suptitle('QQ Plots: Log Returns vs Normal', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('output/qq_plots.png', dpi=150, bbox_inches='tight')
plt.show()


## 5 · Autocorrelation: Returns & Squared Returns

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 7))

pairs = [
    (calm_ret,     'Calm Returns',          axes[0, 0], '#2563eb'),
    (stressed_ret, 'Stressed Returns',       axes[0, 1], '#dc2626'),
    (calm_ret**2,  'Calm Sq. Returns',       axes[1, 0], '#2563eb'),
    (stressed_ret**2,'Stressed Sq. Returns', axes[1, 1], '#dc2626'),
]

for ret, title, ax, color in pairs:
    plot_acf(ret, ax=ax, lags=30, color=color,
             vlines_kwargs={'colors': color},
             alpha=0.05, zero=False)
    ax.set_title(title, fontweight='bold', fontsize=10)
    ax.set_xlabel('Lag (days)')
    ax.set_ylabel('ACF')
    ax.axhline(0, color='grey', lw=0.5)
    # annotate first significant lag
    n = len(ret)
    bound = 1.96 / np.sqrt(n)
    ax.axhline( bound, color='grey', lw=0.6, ls='--')
    ax.axhline(-bound, color='grey', lw=0.6, ls='--')

plt.suptitle('Autocorrelation: Returns (top) and Squared Returns (bottom)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('output/autocorrelation.png', dpi=150, bbox_inches='tight')
plt.show()


## 6 · Regime Parameter Comparison

In [ ]:
p_calm     = params[params.regime == 'calm'].iloc[0]
p_stressed = params[params.regime == 'stressed'].iloc[0]

labels  = [r'$\gamma$ (fund. pull)', r'$\beta$ (momentum)', r'$\sigma_v$ (fund. vol)',
           r'$\sigma_n$ (noise vol)', r'$g$ (drift)']
calm_vals     = [p_calm.gamma,     p_calm.beta,     p_calm.sigma_v,
                 p_calm.sigma_n,   p_calm.g]
stressed_vals = [p_stressed.gamma, p_stressed.beta, p_stressed.sigma_v,
                 p_stressed.sigma_n, p_stressed.g]

x = np.arange(len(labels))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 4))
b1 = ax.bar(x - width/2, calm_vals,     width, label='Calm',     color='#2563eb', alpha=0.8)
b2 = ax.bar(x + width/2, stressed_vals, width, label='Stressed', color='#dc2626', alpha=0.8)
ax.bar_label(b1, fmt='%.4f', fontsize=8, padding=2)
ax.bar_label(b2, fmt='%.4f', fontsize=8, padding=2)
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=11)
ax.set_ylabel('Parameter Value')
ax.set_title('Calibrated Parameter Comparison: Calm vs Stressed',
             fontweight='bold', fontsize=12)
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('output/parameter_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# Print % change table
print('\nParameter changes (stressed vs calm):')
for lbl, cv, sv in zip(labels, calm_vals, stressed_vals):
    pct = 100 * (sv - cv) / (abs(cv) + 1e-12)
    print(f'  {lbl:30s}  calm={cv:.5f}  stressed={sv:.5f}  Δ={pct:+.1f}%')


## 7 · Volatility Clustering (Rolling Volatility)

In [ ]:
window = 10

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=False)

for ax, ret, df_mis, regime, color in [
    (axes[0], calm_ret,     calm_mis,     'Calm',     '#2563eb'),
    (axes[1], stressed_ret, stressed_mis, 'Stressed', '#dc2626'),
]:
    roll_vol = pd.Series(ret).rolling(window).std() * np.sqrt(252)
    ax.plot(roll_vol, color=color, lw=1.2, label=f'{window}-day rolling vol (ann.)')
    ax2 = ax.twinx()
    ax2.plot(np.abs(df_mis.values[1:]), color='grey', lw=0.8,
             alpha=0.6, label='|Mispricing|')
    ax.set_title(f'{regime}', fontweight='bold')
    ax.set_ylabel('Annualised Volatility')
    ax2.set_ylabel('|Mispricing|', color='grey')
    ax.legend(loc='upper left', fontsize=8)
    ax2.legend(loc='upper right', fontsize=8)

axes[1].set_xlabel('Trading Day')
plt.suptitle('Rolling Volatility vs Mispricing Magnitude',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('output/volatility_clustering.png', dpi=150, bbox_inches='tight')
plt.show()


## 8 · Summary Statistics Table

In [ ]:
rows = []
for ret, mis, regime in [
    (calm_ret,     calm_mis,     'Calm'),
    (stressed_ret, stressed_mis, 'Stressed'),
]:
    rows.append({
        'Regime'           : regime,
        'N obs'            : len(ret),
        'Mean return'      : f'{ret.mean():.5f}',
        'Std return'       : f'{ret.std():.5f}',
        'Skewness'         : f'{stats.skew(ret):.3f}',
        'Excess kurtosis'  : f'{stats.kurtosis(ret):.3f}',
        'JB p-value'       : f'{stats.jarque_bera(ret)[1]:.4f}',
        'Mean |mispricing|': f'{np.abs(mis).mean():.5f}',
        'Std mispricing'   : f'{mis.std():.5f}',
        'Durbin-Watson'    : f'{durbin_watson(ret):.3f}',
    })

summary = pd.DataFrame(rows).set_index('Regime').T
display(summary)
summary.to_csv('output/summary_statistics.csv')
print('Saved to output/summary_statistics.csv')
